# Check that the interval survives every boost, exactly

*Adversarial Mathematics From First Principles*, [chapter 18](https://programmer.ie/books/math/18-chapter/) · [worked example](https://programmer.ie/examples/math/18-chapter/)

Run the cells in order. The program is a small, self-contained demonstration of the chapter's mathematics; the output saved with it is what it printed when this notebook was built.

The interval t² − x² is the quantity every inertial frame agrees on. This
program boosts four events at speeds with rational γ (3/5, 5/13, −8/17), so
nothing is rounded, and checks the interval each time. It then shows the
light-cone scaling, which events can change order, and that two boosts compose
into one.

## Run it

Standard library only; nothing to install.

In [1]:
"""The Lorentz transformation and the invariant interval, exactly (standard library only)."""
from fractions import Fraction as F


def gamma(v):
    """gamma for speeds where 1 - v^2 is a rational square, so everything stays exact."""
    num, den = (1 - v * v).numerator, (1 - v * v).denominator
    r_num, r_den = round(num ** 0.5), round(den ** 0.5)
    assert r_num * r_num == num and r_den * r_den == den, "choose v with rational gamma"
    return 1 / F(r_num, r_den)


def boost(v):
    g = gamma(v)
    return lambda t, x: (g * (t - v * x), g * (x - v * t))


def interval(e):
    return e[0] ** 2 - e[1] ** 2


events = [(F(5), F(3)), (F(1), F(2)), (F(2), F(2)), (F(-3), F(7, 2))]
print("The interval t^2 - x^2 before and after boosts at v = 3/5, 5/13, -8/17:")
for e in events:
    after = [interval(boost(v)(*e)) for v in (F(3, 5), F(5, 13), F(-8, 17))]
    kind = "timelike" if interval(e) > 0 else "spacelike" if interval(e) < 0 else "lightlike"
    print(f"  ({e[0]}, {e[1]}): {interval(e)} -> {', '.join(map(str, after))}   {kind}")

print("\nLight-cone coordinates u = t - x, w = t + x are scaled by k and 1/k:")
v = F(3, 5)
for e in events[:2]:
    t2, x2 = boost(v)(*e)
    print(f"  ({e[0]}, {e[1]}): u {e[0] - e[1]} -> {t2 - x2}, w {e[0] + e[1]} -> {t2 + x2}, "
          f"ratios {(t2 - x2) / (e[0] - e[1])} and {(t2 + x2) / (e[0] + e[1])}")

print("\nThe order of events can change only for spacelike separation:")
for e in [(F(1), F(2)), (F(5), F(3))]:
    flipped = [v for v in (F(3, 5), F(5, 13), F(12, 13), F(-3, 5)) if boost(v)(*e)[0] < 0]
    print(f"  O then ({e[0]}, {e[1]}), {'spacelike' if interval(e) < 0 else 'timelike'}: "
          f"order reversed for v in {[str(x) for x in flipped] or 'none of the tested speeds'}")

print("\nComposing boosts: 3/5 then 5/13 is one boost at (u + w)/(1 + u w)")
u, w = F(3, 5), F(5, 13)
both = lambda t, x: boost(w)(*boost(u)(t, x))
single = boost((u + w) / (1 + u * w))
print(f"  composite speed {(u + w) / (1 + u * w)}; same on every test event: "
      f"{all(both(*e) == single(*e) for e in events)}")

print("\nA control: the Galilean shear does not preserve the interval")
shear = lambda t, x: (t, x - F(3, 5) * t)
print(f"  (5, 3): {interval((F(5), F(3)))} -> {interval(shear(F(5), F(3)))}")

The interval t^2 - x^2 before and after boosts at v = 3/5, 5/13, -8/17:
  (5, 3): 16 -> 16, 16, 16   timelike
  (1, 2): -3 -> -3, -3, -3   spacelike
  (2, 2): 0 -> 0, 0, 0   lightlike
  (-3, 7/2): -13/4 -> -13/4, -13/4, -13/4   spacelike

Light-cone coordinates u = t - x, w = t + x are scaled by k and 1/k:
  (5, 3): u 2 -> 4, w 8 -> 4, ratios 2 and 1/2
  (1, 2): u -1 -> -2, w 3 -> 3/2, ratios 2 and 1/2

The order of events can change only for spacelike separation:
  O then (1, 2), spacelike: order reversed for v in ['3/5', '12/13']
  O then (5, 3), timelike: order reversed for v in none of the tested speeds

Composing boosts: 3/5 then 5/13 is one boost at (u + w)/(1 + u w)
  composite speed 4/5; same on every test event: True

A control: the Galilean shear does not preserve the interval
  (5, 3): 16 -> 25


## What the output shows

The interval is the same in every frame: 16 stays 16, −3 stays −3, 0 stays 0.
In light-cone coordinates a boost is a scaling by k and 1/k (here 2 and 1/2).
Only the spacelike event changes order relative to O, and only for some
speeds; the timelike one never does. Boosts at 3/5 and 5/13 compose to one at
4/5. The Galilean shear, as a control, changes 16 to 25.

## Not shown

- Invariance is checked on four events at three speeds; the algebraic proof is γ²(1 − v²) = 1, which the chapter derives.
- Only speeds with rational γ are used, to stay exact; other speeds need surds or symbolic algebra.

## Challenge it

Examine the program's checks of t² − x² under boosts. Prove the invariance
algebraically from γ²(1 − v²) = 1, derive the composition law
(u + w)/(1 + uw), and show that a timelike separation can never change order.
Then find the speed at which the spacelike event (1, 2) becomes simultaneous
with O.